# Oversampling

This notebook demonstrates the process of balancing class distributions in the dataset through **oversampling**.  
The procedure works as follows:  

- A test set containing multiple classes is loaded.  
- For classes with fewer samples, additional images are generated by randomly selecting existing images from that class and applying random rotations between **1° and 359°**.  
- Oversampling continues until each class reaches the size of the largest class in the dataset.  

This ensures a more balanced dataset, which helps improve model training and performance.  


In [1]:
import os
import shutil
import random
from PIL import Image
from zipfile import ZipFile

In [5]:
src_dir = './train'         
dst_dir = './train_oversampling' 

class_counts = {}
for class_name in os.listdir(src_dir):
    class_path = os.path.join(src_dir, class_name)
    if not os.path.isdir(class_path):
        continue
    images = [f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))]
    class_counts[class_name] = len(images)

N = max(class_counts.values())
print(f"Maximum class size (N) = {N}")
print("Original class counts:", class_counts)

if os.path.exists(dst_dir):
    shutil.rmtree(dst_dir)
os.makedirs(dst_dir)

for class_name in class_counts:
    print (class_name,class_counts[class_name])
    img_no=class_counts[class_name]
    src_class_dir = os.path.join(src_dir, class_name)
    dst_class_dir = os.path.join(dst_dir, class_name)
    os.makedirs(dst_class_dir, exist_ok=True)

    images = os.listdir(src_class_dir)
    random.shuffle(images)
    selected_images = images
    for img_name in selected_images:
        src_path = os.path.join(src_class_dir, img_name)
        dst_path = os.path.join(dst_class_dir, img_name)
        shutil.copy(src_path, dst_path)
    if img_no<N:
        count=0
        for i in range(N-img_no):
            count+=1
            random_image_name = random.choice(images)
            image_path = os.path.join(src_class_dir, random_image_name)
            image = Image.open(image_path)
            original_size=image.size
            #angle = random.choice([90,180,270])
            angle = random.randint(1,359)
            rotated_image = image.rotate(angle, expand=True)
            rotated_width, rotated_height = rotated_image.size
            orig_width, orig_height = original_size

            left = (rotated_width - orig_width) // 2
            top = (rotated_height - orig_height) // 2
            right = left + orig_width
            bottom = top + orig_height

            cropped = rotated_image.crop((left, top, right, bottom))
            output_path = os.path.join(dst_class_dir, f"rotated_{angle}_{count}_{random_image_name}")
            cropped.save(output_path)
        print (f'rotated {count} images, total of {count+img_no}')

print(f"Copied {N} images per class to {dst_dir}")

Maximum class size (N) = 2330
Original class counts: {'basophil': 852, 'eosinophil': 2181, 'erythroblast': 1085, 'ig': 2026, 'lymphocyte': 849, 'monocyte': 993, 'neutrophil': 2330, 'platelet': 1643}
basophil 852
rotated 1478 images, total of 2330
eosinophil 2181
rotated 149 images, total of 2330
erythroblast 1085
rotated 1245 images, total of 2330
ig 2026
rotated 304 images, total of 2330
lymphocyte 849
rotated 1481 images, total of 2330
monocyte 993
rotated 1337 images, total of 2330
neutrophil 2330
platelet 1643
rotated 687 images, total of 2330
Copied 2330 images per class to ./train_oversampling


In [2]:
def zip_multiple_folders(folder_paths, output_zip_path):
    with ZipFile(output_zip_path, 'w') as zipf:
        for folder_path in folder_paths:
            base_folder_name = os.path.basename(folder_path.rstrip("/"))
            for root, dirs, files in os.walk(folder_path):
                for file in files:
                    file_path = os.path.join(root, file)
                    # Create a relative path that keeps the folder name
                    arcname = os.path.join(base_folder_name, os.path.relpath(file_path, start=folder_path))
                    zipf.write(file_path, arcname)
    print(f"Zipped {len(folder_paths)} folders into '{output_zip_path}'")

# Example usage:
folders_to_zip = ['./train_oversampling', './test','./val']
zip_multiple_folders(folders_to_zip, 'train_test_val_data.zip')

Zipped 3 folders into 'train_test_val_data.zip'
